# Day 41 — Practice 1: RDD Operations

**Tujuan:** Memahami konsep RDD (Resilient Distributed Dataset) — fondasi Spark sebelum DataFrame API.

```
RDD = Resilient (tahan gagal) + Distributed (tersebar di cluster) + Dataset (kumpulan data)
```

**Yang dipraktikkan:**
1. Buat RDD dari berbagai sumber
2. Transformasi (lazy): `map`, `filter`, `flatMap`, `groupByKey`, `reduceByKey`
3. Action (trigger eksekusi): `collect`, `count`, `take`, `reduce`, `saveAsTextFile`
4. Pair RDD (key-value)
5. RDD dari data AdventureWorks nyata

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('RDD-Practice') \
    .getOrCreate()

sc = spark.sparkContext  # SparkContext — entry point untuk RDD

print(f'Spark {spark.version} | Default parallelism: {sc.defaultParallelism}')

## 1. Buat RDD dari Berbagai Sumber

In [ ]:
# --- 1.1 Dari Python list ---
rdd_numbers = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print(f'rdd_numbers: partitions={rdd_numbers.getNumPartitions()}, type={type(rdd_numbers)}')

# --- 1.2 Dengan jumlah partisi ditentukan ---
rdd_4parts = sc.parallelize(range(20), numSlices=4)
print(f'rdd_4parts : partitions={rdd_4parts.getNumPartitions()}')

# --- 1.3 Dari file teks di HDFS ---
# rdd_file = sc.textFile('hdfs://namenode:9000/datalake/raw/employees.csv')

# --- 1.4 Dari DataFrame (konversi) ---
df_sample = spark.createDataFrame([(1,'A',100),(2,'B',200),(3,'A',150)], ['id','cat','val'])
rdd_from_df = df_sample.rdd
print(f'rdd_from_df: partitions={rdd_from_df.getNumPartitions()}')
print(f'Sample row : {rdd_from_df.first()}')

## 2. Transformasi Dasar (LAZY — belum dieksekusi!)

In [ ]:
# map: terapkan fungsi ke setiap elemen
rdd_doubled  = rdd_numbers.map(lambda x: x * 2)
rdd_squared  = rdd_numbers.map(lambda x: x ** 2)
rdd_labeled  = rdd_numbers.map(lambda x: (x, 'even' if x % 2 == 0 else 'odd'))

# filter: ambil elemen yang memenuhi kondisi
rdd_evens    = rdd_numbers.filter(lambda x: x % 2 == 0)
rdd_big      = rdd_numbers.filter(lambda x: x > 5)

# flatMap: setiap elemen bisa menghasilkan 0 atau lebih elemen baru
rdd_words    = sc.parallelize(['spark is fast', 'hadoop is storage', 'hive is warehouse'])
rdd_split    = rdd_words.flatMap(lambda s: s.split(' '))

# distinct: hilangkan duplikasi
rdd_dup      = sc.parallelize([1,1,2,3,3,4,5,5])
rdd_unique   = rdd_dup.distinct()

print('Transformasi didefinisikan (belum dieksekusi — lazy evaluation)')

## 3. Actions (Trigger Eksekusi)

In [ ]:
print('=== collect() — ambil semua ke driver ===')
print('doubled :', rdd_doubled.collect())
print('evens   :', rdd_evens.collect())
print('labeled :', rdd_labeled.collect())

print('\n=== take(n) — ambil n elemen pertama ===')
print('words (split) take(5):', rdd_split.take(5))

print('\n=== count() — hitung jumlah elemen ===')
print(f'total numbers : {rdd_numbers.count()}')
print(f'total evens   : {rdd_evens.count()}')
print(f'total unique  : {rdd_unique.count()}')

print('\n=== reduce() — agregasi jadi 1 nilai ===')
print(f'sum  : {rdd_numbers.reduce(lambda a, b: a + b)}')
print(f'max  : {rdd_numbers.max()}')
print(f'min  : {rdd_numbers.min()}')

print('\n=== first() — elemen pertama ===')
print(f'first: {rdd_numbers.first()}')

## 4. Pair RDD (Key-Value RDD)

Pair RDD adalah RDD dengan elemen berupa tuple `(key, value)`.
Ini fondasi dari **shuffle operations** seperti groupBy dan join.

In [ ]:
# Buat Pair RDD
sales_data = sc.parallelize([
    ('Jakarta',  1500000),
    ('Surabaya',  800000),
    ('Jakarta',  2200000),
    ('Bandung',   600000),
    ('Surabaya', 1100000),
    ('Jakarta',   900000),
    ('Bandung',  1400000),
    ('Surabaya',  750000),
])

# reduceByKey: agregasi value per key (efisien, combine di tiap partisi dulu)
total_per_city = sales_data.reduceByKey(lambda a, b: a + b)
print('=== Total per kota (reduceByKey) ===')
for city, total in total_per_city.sortBy(lambda x: x[1], ascending=False).collect():
    print(f'  {city:12s}: Rp {total:>12,}')

# groupByKey: kumpulkan semua value per key (lebih boros memori)
grouped = sales_data.groupByKey()
print('\n=== Semua transaksi per kota (groupByKey) ===')
for city, values in grouped.collect():
    vals = list(values)
    print(f'  {city:12s}: {[f"Rp{v:,}" for v in vals]}')

# countByKey: hitung jumlah transaksi per kota
print('\n=== Jumlah transaksi per kota (countByKey) ===')
for city, count in sales_data.countByKey().items():
    print(f'  {city:12s}: {count} transaksi')

In [ ]:
# mapValues: transformasi value tanpa mengubah key
# Ubah revenue dari rupiah ke juta
in_million = sales_data.mapValues(lambda v: round(v / 1_000_000, 2))
print('=== Revenue dalam juta (mapValues) ===')
print(in_million.collect())

# flatMapValues: 1 value bisa jadi banyak
products_per_city = sc.parallelize([
    ('Jakarta',  ['Laptop', 'Monitor', 'Keyboard']),
    ('Surabaya', ['Mouse', 'Headset']),
])
flat = products_per_city.flatMapValues(lambda v: v)
print('\n=== flatMapValues (1 row per produk) ===')
print(flat.collect())

## 5. Word Count — Contoh Klasik MapReduce

In [ ]:
text_rdd = sc.parallelize([
    'apache spark is a fast unified analytics engine',
    'apache hadoop is a distributed storage system',
    'apache hive is a data warehouse built on hadoop',
    'spark runs on hadoop yarn kubernetes and standalone',
    'hive stores metadata in a metastore connected to hadoop',
])

word_count = text_rdd \
    .flatMap(lambda line: line.split(' '))    \
    .filter(lambda w: len(w) > 2)             \
    .map(lambda w: (w, 1))                    \
    .reduceByKey(lambda a, b: a + b)          \
    .sortBy(lambda x: x[1], ascending=False)   

print('=== Top 15 kata ===')
for word, count in word_count.take(15):
    bar = '█' * count
    print(f'  {word:12s} {count:2d} {bar}')

## 6. RDD dari Data AdventureWorks Nyata

In [ ]:
# Baca dari HDFS (data yang sudah di-extract di notebook sebelumnya)
HDFS_BASE = 'hdfs://namenode:9000/datalake/raw'

# Baca Parquet sebagai DataFrame lalu convert ke RDD
df_orders = spark.read.parquet(f'{HDFS_BASE}/sales/salesorderheader')
df_products = spark.read.parquet(f'{HDFS_BASE}/production/product')

print(f'Orders  : {df_orders.count():,} rows')
print(f'Products: {df_products.count():,} rows')

In [ ]:
# Convert ke RDD dan analisis dengan RDD operations
orders_rdd = df_orders.select('salesorderid', 'orderdate', 'totaldue', 'order_year').rdd

# Hitung total revenue per tahun menggunakan RDD
revenue_by_year = orders_rdd \
    .map(lambda row: (row['order_year'], float(row['totaldue'] or 0))) \
    .reduceByKey(lambda a, b: a + b) \
    .sortByKey()

print('=== Revenue per Tahun (via RDD) ===')
for year, revenue in revenue_by_year.collect():
    print(f'  {year}: ${revenue:>15,.2f}')

In [ ]:
# Produk dengan harga di atas average menggunakan RDD filter
products_rdd = df_products.select('name', 'listprice', 'color', 'productline').rdd

avg_price = df_products.agg(F.avg('listprice')).collect()[0][0]
print(f'Average list price: ${avg_price:.2f}')

premium = products_rdd \
    .filter(lambda r: r['listprice'] and r['listprice'] > avg_price) \
    .map(lambda r: (r['name'], r['listprice'], r['color'])) \
    .sortBy(lambda x: x[1], ascending=False)

print(f'\nProduk premium ({premium.count()} produk):')
for name, price, color in premium.take(10):
    print(f'  {name:40s} ${price:>8.2f}  {color or "-"}')

## 7. Perbandingan: RDD vs DataFrame

Kapan pakai yang mana?

In [ ]:
import time

# Operasi yang sama: hitung total revenue per tahun
# --- Cara 1: RDD ---
t0 = time.time()
result_rdd = orders_rdd \
    .map(lambda r: (r['order_year'], float(r['totaldue'] or 0))) \
    .reduceByKey(lambda a, b: a + b) \
    .collect()
t_rdd = time.time() - t0

# --- Cara 2: DataFrame ---
t0 = time.time()
result_df = df_orders.groupBy('order_year') \
    .agg(F.sum('totaldue').alias('total')) \
    .collect()
t_df = time.time() - t0

print(f'RDD   : {t_rdd:.2f}s')
print(f'DataFrame: {t_df:.2f}s')
print()
print('Kapan pakai RDD:')
print('  - Custom complex transformations yang sulit di-express dengan DataFrame API')
print('  - Data non-tabular (raw text, binary, JSON nested kompleks)')
print('  - Low-level control atas partisi dan distribusi data')
print()
print('Kapan pakai DataFrame:')
print('  - Query SQL-like (filter, join, groupBy, aggregate)')
print('  - Structured/semi-structured data (Parquet, CSV, JSON)')
print('  - Production workloads — lebih optimal karena Catalyst optimizer')

## Summary

Yang sudah dipraktikkan:
- ✅ Buat RDD dari list, range, DataFrame
- ✅ Transformasi: `map`, `filter`, `flatMap`, `distinct`, `sortBy`
- ✅ Actions: `collect`, `take`, `count`, `reduce`, `first`, `max`, `min`
- ✅ Pair RDD: `reduceByKey`, `groupByKey`, `countByKey`, `mapValues`, `flatMapValues`
- ✅ Word Count — implementasi MapReduce paradigm dengan RDD
- ✅ RDD dari data AdventureWorks nyata di HDFS
- ✅ Perbandingan RDD vs DataFrame

**Next →** `02_hive_to_spark.ipynb` — Koneksi Hive ke Spark dan transformasi data.